# 01 — Data Profiling

This notebook answers **"what exactly are we looking at"**: the
structure, completeness, uniqueness, validity, and internal
consistency of the raw data.

It deliberately does **not** look for relationships with the target
or between predictors, and it does **not** visualize anything — that
is the job of `02_eda.ipynb`. The test applied throughout: a check
belongs here if it describes a column on its own terms, or checks
whether two columns that *should* structurally agree actually do. The
moment a check asks "does this relate to `posted_rate`" — correlation
with the target, the shape of the target's distribution and whether
it's worth transforming — it's a hypothesis about what matters, and
that's EDA's job, not profiling's, no matter how tempting it is to
peek early.

In [1]:
import logging
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src import config, data, profiling

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## Load the raw files

No cleaning, no type coercion beyond what pandas infers by default —
`src/data.py` reads these exactly as they arrived.

In [3]:
train_test = data.load_train_test()
validation = data.load_validation()
val_template = data.load_validation_template()
december = data.load_december()

INFO src.data: Loading C:\dev\Spotter-ML-Engineer-Assessment\data\raw\train_test.csv
INFO src.data: Loaded train_test.csv: 48000 rows, 14 columns
INFO src.data: Loading C:\dev\Spotter-ML-Engineer-Assessment\data\raw\validation.csv
INFO src.data: Loaded validation.csv: 12000 rows, 13 columns
INFO src.data: Loading C:\dev\Spotter-ML-Engineer-Assessment\data\raw\validation_predictions_template.csv
INFO src.data: Loaded validation_predictions_template.csv: 12000 rows, 2 columns
INFO src.data: Loading C:\dev\Spotter-ML-Engineer-Assessment\data\raw\december_chart_inputs.csv
INFO src.data: Loaded december_chart_inputs.csv: 31 rows, 7 columns


## Structure: shape, columns, dtypes, a sample of rows

In [4]:
for name, df in [
    ("train_test", train_test),
    ("validation", validation),
    ("validation_template", val_template),
    ("december", december),
]:
    info = profiling.describe_structure(df, name)
    print(f"--- {info['name']} ---")
    print(f"shape: {info['n_rows']} rows x {info['n_columns']} columns")
    print(f"columns: {info['columns']}")
    print()

INFO src.profiling: Describing structure of train_test
INFO src.profiling: Describing structure of validation
INFO src.profiling: Describing structure of validation_template
INFO src.profiling: Describing structure of december


--- train_test ---
shape: 48000 rows x 14 columns
columns: ['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate']

--- validation ---
shape: 12000 rows x 13 columns
columns: ['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal']

--- validation_template ---
shape: 12000 rows x 2 columns
columns: ['load_id', 'predicted_rate']

--- december ---
shape: 31 rows x 7 columns
columns: ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']



`validation` has one fewer column than `train_test` — no `posted_rate`,
which is expected: it's the file Spotter scores us on after
submission, so the target is withheld. `validation_template` is just
the two columns we need to fill in for that file. `december` has the
fewest columns of all (7, no `load_id`, no target) — its row identity
is `date`, not an id column, and several columns present everywhere
else (both coordinate pairs, `market_index`, `quote_signal`) are
absent entirely, not just empty. That's a structural fact worth
carrying into cleaning and feature engineering: whatever model
produces the December predictions has to run without those columns.

In [5]:
train_test.head()

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [6]:
train_test.dtypes

load_id          object
pickup           object
delivery         object
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment        object
weight          float64
date             object
market_index    float64
quote_signal    float64
posted_rate     float64
dtype: object

All numeric columns load as `float64` except `date`, which loads as
`object` (string) — `src/data.py` does zero type coercion by design,
so nothing here has been parsed as a date yet. `check_date_coverage`
below parses it internally just to run its check; converting `date`
to an actual datetime dtype in the DataFrame itself is a deterministic
formatting fix that belongs in cleaning, not here.

Every column is either numeric (`float64`) or categorical (`object`
strings: `load_id`, `pickup`, `delivery`, `equipment`, `date`), and
the target, `posted_rate`, is continuous — confirming this is a
regression problem, not classification.

## Missingness

Which columns have missing values, and how much.

In [7]:
profiling.check_missingness(train_test)

INFO src.profiling: Checking missingness across 14 columns
INFO src.profiling: Columns with missing values: 2


,n_missing,pct_missing
market_index,374,0.779
weight,300,0.625
delivery,0,0.000
pickup_lat,0,0.000
load_id,0,0.000
pickup,0,0.000
delivery_lat,0,0.000
pickup_lon,0,0.000
distance,0,0.000
delivery_lon,0,0.000


In [8]:
profiling.check_missingness(validation)

INFO src.profiling: Checking missingness across 13 columns
INFO src.profiling: Columns with missing values: 2


,n_missing,pct_missing
market_index,249,2.075
weight,165,1.375
delivery,0,0.000
pickup,0,0.000
load_id,0,0.000
pickup_lon,0,0.000
pickup_lat,0,0.000
delivery_lat,0,0.000
delivery_lon,0,0.000
equipment,0,0.000


In [9]:
profiling.check_missingness(december)

INFO src.profiling: Checking missingness across 7 columns
INFO src.profiling: Columns with missing values: 1


,n_missing,pct_missing
predicted_rate,31,100.0
delivery,0,0.0
pickup,0,0.0
distance,0,0.0
equipment,0,0.0
weight,0,0.0
date,0,0.0


Two columns are affected in both `train_test` and `validation`:
`market_index` and `weight`. Both are under 1% missing in
`train_test`, but the rate is meaningfully higher in `validation`
(`market_index` 2.08% vs. 0.78%, `weight` 1.38% vs. 0.63%) — worth a
sentence in the report even without knowing why. `december`'s only
"missing" column is `predicted_rate`, which is 100% missing by
design — that's the column we're filling in, not a data quality
issue.

We deliberately check missingness on `validation` here, even though
no rows will ever be dropped from it — every row needs a prediction
regardless of what's missing, per the assessment brief. The reason
to check anyway is that this describes what we're evaluated on: if
`validation` had, say, 20% missingness in some column, that would be
a materially different data-quality situation than under 1%, and
would push the choice of imputation strategy in cleaning. Describing
it is profiling; deciding what to do about it is cleaning.

## Uniqueness and duplicate rows

In [10]:
profiling.check_uniqueness(train_test, config.ID_COL)

INFO src.profiling: Checking uniqueness for id_col=load_id
INFO src.profiling: Uniqueness result: {'id_col': 'load_id', 'n_rows': 48000, 'n_unique_ids': 48000, 'is_unique': True, 'n_duplicate_ids': 0, 'n_duplicate_rows': 0}


{'id_col': 'load_id',
 'n_rows': 48000,
 'n_unique_ids': 48000,
 'is_unique': True,
 'n_duplicate_ids': 0,
 'n_duplicate_rows': 0}

In [11]:
profiling.check_uniqueness(validation, config.ID_COL)

INFO src.profiling: Checking uniqueness for id_col=load_id
INFO src.profiling: Uniqueness result: {'id_col': 'load_id', 'n_rows': 12000, 'n_unique_ids': 12000, 'is_unique': True, 'n_duplicate_ids': 0, 'n_duplicate_rows': 0}


{'id_col': 'load_id',
 'n_rows': 12000,
 'n_unique_ids': 12000,
 'is_unique': True,
 'n_duplicate_ids': 0,
 'n_duplicate_rows': 0}

`load_id` is fully unique in both files, and there are zero exact
duplicate rows in either. No de-duplication needed in cleaning.

## Domain / range sanity checks

Physically impossible values: non-positive distance, weight, or
posted_rate; out-of-range coordinates; pickup equal to delivery. Only
checks the columns actually present in each file.

In [12]:
profiling.check_domain_ranges(train_test)

INFO src.profiling: Running domain/range checks
INFO src.profiling: Domain/range checks complete: 8 checks run


,check,n_flagged,description
0,distance_non_positive,0,Rows where distance <= 0 (physically impossible)
1,weight_non_positive,292,Rows where weight <= 0 (physically impossible)
2,posted_rate_non_positive,0,Rows where posted_rate <= 0 (physically imposs...
3,pickup_lat_out_of_range,0,"Rows where pickup_lat is outside [-90, 90]"
4,delivery_lat_out_of_range,0,"Rows where delivery_lat is outside [-90, 90]"
5,pickup_lon_out_of_range,0,"Rows where pickup_lon is outside [-180, 180]"
6,delivery_lon_out_of_range,0,"Rows where delivery_lon is outside [-180, 180]"
7,pickup_equals_delivery,0,Rows where pickup and delivery are the same city


In [13]:
profiling.check_domain_ranges(validation)

INFO src.profiling: Running domain/range checks
INFO src.profiling: Domain/range checks complete: 7 checks run


,check,n_flagged,description
0,distance_non_positive,0,Rows where distance <= 0 (physically impossible)
1,weight_non_positive,145,Rows where weight <= 0 (physically impossible)
2,pickup_lat_out_of_range,0,"Rows where pickup_lat is outside [-90, 90]"
3,delivery_lat_out_of_range,0,"Rows where delivery_lat is outside [-90, 90]"
4,pickup_lon_out_of_range,0,"Rows where pickup_lon is outside [-180, 180]"
5,delivery_lon_out_of_range,0,"Rows where delivery_lon is outside [-180, 180]"
6,pickup_equals_delivery,0,Rows where pickup and delivery are the same city


In [14]:
profiling.check_domain_ranges(december)

INFO src.profiling: Running domain/range checks
INFO src.profiling: Domain/range checks complete: 3 checks run


,check,n_flagged,description
0,distance_non_positive,0,Rows where distance <= 0 (physically impossible)
1,weight_non_positive,0,Rows where weight <= 0 (physically impossible)
2,pickup_equals_delivery,0,Rows where pickup and delivery are the same city


Every check comes back zero except `weight_non_positive`: 292 rows in
`train_test` (0.61%) and 145 rows in `validation` (1.21%) have
`weight <= 0`. Everything else — distance, posted_rate, coordinate
bounds, pickup equal to delivery — is clean in every file, including
`december`.

`weight_non_positive` and the `weight` missingness above are separate
issues, not the same rows: only one row in the entire dataset has
both problems. What the negative values actually look like (whether
they're a plausible sign-flip or something else) is a cleaning
question — addressed with a real visual comparison in
`02_cleaning.ipynb`, not asserted here.

## Cardinality of categorical columns

In [15]:
cardinality = profiling.check_cardinality(train_test, ["pickup", "delivery", "equipment"])
for col, info in cardinality.items():
    print(f"--- {col}: {info['n_unique']} unique values ---")
    print(info["top_values"])
    print()

INFO src.profiling: Checking cardinality for columns: ['pickup', 'delivery', 'equipment']


--- pickup: 64 unique values ---
pickup
Oklahoma City    1242
Lexington        1209
Bakersfield      1193
Fort Wayne       1170
Hartford         1150
Richmond         1140
Nashville        1124
Phoenix          1121
Baton Rouge      1115
Mobile           1094
Name: count, dtype: int64

--- delivery: 64 unique values ---
delivery
Lexington        1197
Fort Wayne       1176
Baton Rouge      1167
Bakersfield      1156
Hartford         1143
Oklahoma City    1140
Richmond         1109
Atlanta          1096
Phoenix          1090
Mobile           1089
Name: count, dtype: int64

--- equipment: 3 unique values ---
equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64



64 distinct pickup cities, 64 distinct delivery cities, only 3
equipment types (Dry Van 27,202 / Reefer 12,045 / Flatbed 8,753).
The city cardinality matters directly for feature engineering: one-hot
encoding both pickup and delivery would add roughly 120+ columns
against ~48,000 rows, which is a real cost to weigh against
alternatives (target encoding, frequency encoding, or a model with
native categorical support) once we're choosing models — a decision
for feature engineering, not resolved here. Equipment's low
cardinality (3 values) makes one-hot encoding for it essentially free
by comparison.

## Descriptive statistics for numeric columns

Plain `.describe()` output — no comment on shape or skew, no
visualization. Whether any of this needs a transform is an EDA
question, not a profiling one.

In [16]:
numeric_cols = ["distance", "weight", "market_index", "quote_signal", "posted_rate"]
profiling.describe_numeric(train_test, numeric_cols)

INFO src.profiling: Describing numeric columns: ['distance', 'weight', 'market_index', 'quote_signal', 'posted_rate']


,count,mean,std,min,25%,50%,75%,max
distance,48000.0,1135.856654,728.564416,70.00000,550.40000,953.30000,1645.525000,3439.80000
weight,47700.0,31028.844004,9391.440620,-47500.00000,25800.00000,31436.50000,37018.000000,47500.00000
market_index,47626.0,1.083387,0.168091,0.67639,0.94967,1.05580,1.219590,1.46778
quote_signal,48000.0,2.062468,0.291391,0.69228,1.89103,2.05575,2.221685,3.61035
posted_rate,48000.0,2373.980682,1486.493245,57.22000,1251.55500,2030.76000,3330.750000,25533.00000


`count` is below 48,000 for `weight` (47,700) and `market_index`
(47,626) because pandas' `.describe()` silently excludes NaNs — these
are the same missing-value counts already reported above, not a new
finding.

One structural fact worth naming here, without going anywhere near
shape or skew: these columns sit on very different numeric scales —
`distance` ranges into the thousands, `posted_rate` into the tens of
thousands, while `market_index` and `quote_signal` stay under 4. That
alone (different units, different ranges) means any scale-sensitive
model (e.g. a linear model) will need some form of scaling in the
pipeline. That inference follows directly from the units, not from a
pattern in the data — so it's fair to note here. Which specific
scaler, and whether a log-transform is also warranted, depends on the
*shape* of these distributions, which is what `02_eda.ipynb` looks at.

Also visible here, and already flagged under domain checks:
`weight`'s `min` is deeply negative (-47,500) purely because of the
sign-flip issue — this table will look different once that's
resolved in cleaning.

## Distance vs. coordinates: internal consistency check

Does the given `distance` column agree with the great-circle distance
implied by the pickup/delivery coordinates? A question about whether
two columns that should structurally relate actually do — not a
question about what predicts the target.

In [17]:
distance_check = profiling.check_distance_consistency(train_test)
print("correlation between distance and great-circle distance:", distance_check["correlation"])
distance_check["ratio_describe"]

INFO src.profiling: Checking distance-vs-coordinates consistency
INFO src.profiling: Distance consistency: correlation=0.9995, 22 rows flagged (ratio > 2x)


correlation between distance and great-circle distance: 0.9995349596237125


count    48000.000000
mean         1.194681
std          0.143541
min          1.058090
25%          1.164765
50%          1.182231
75%          1.204765
max          9.634919
dtype: float64

`distance` and the great-circle distance implied by the coordinates
correlate at 0.9995 — near-perfect agreement across the dataset as a
whole. This correlation is between `distance` and one derived value
computed from *all four* coordinate columns together (the haversine
formula — shortest possible "as the crow flies" distance between two
lat/lon points on a sphere); it says nothing about any single
coordinate column in isolation. The typical ratio of
`distance / great_circle_distance` sits around 1.15–1.20 (see the
25th/50th/75th percentiles above) — road distance running 15–20%
longer than straight-line distance is exactly what real routing looks
like, since roads bend around things a straight line doesn't.

In [18]:
print(f"{len(distance_check['flagged_rows'])} rows flagged (distance more than 2x the great-circle distance)")
distance_check["flagged_rows"][["load_id", "pickup", "delivery", "distance", "great_circle_distance", "ratio"]]

22 rows flagged (distance more than 2x the great-circle distance)


,load_id,pickup,delivery,distance,great_circle_distance,ratio
1464,TR-001465,Lubbock,Austin,70.0,34.46283,2.031174
1914,TR-001915,Lubbock,Austin,70.0,34.46283,2.031174
4140,TR-004141,New Orleans,Shreveport,70.0,7.26524,9.634919
5091,TR-005092,New Orleans,Shreveport,70.0,7.26524,9.634919
6960,TR-006961,New Orleans,Shreveport,70.0,7.26524,9.634919
8589,TR-008590,Austin,Lubbock,70.0,34.46283,2.031174
9348,TR-009349,Shreveport,New Orleans,70.0,7.26524,9.634919
11311,TR-011312,Lubbock,Austin,70.0,34.46283,2.031174
11327,TR-011328,Austin,Lubbock,78.9,34.46283,2.289423
11949,TR-011950,Shreveport,New Orleans,70.0,7.26524,9.634919


22 rows exceed a 2x ratio (a threshold chosen because normal road
routing inflation rarely passes ~1.5–2x even for indirect real
routes) — all concentrated in exactly two lanes, in both directions:
Austin↔Lubbock (10 rows, ratio ~2.0–2.3) and New Orleans↔Shreveport
(12 rows, ratio ~9.6). The more telling detail is what's constant
across all 22 rows: every single one has `distance = 70.0`, despite
Austin↔Lubbock and New Orleans↔Shreveport being genuinely different
real-world separations (implied great-circle distances of ~34.5mi and
~7.3mi respectively). Two different city pairs landing on the
identical distance value isn't a coincidence a real routing process
would produce — this reads as a fallback or default value used
somewhere upstream for these two specific lanes, not ordinary
measurement noise. How to treat it (correct the distance, correct the
coordinates, or flag the rows) is a cleaning decision, not resolved
here.

## Coordinate stability per city

A separate question from the check above: is latitude/longitude a
stable, deterministic function of city name — does every occurrence
of, say, "Richmond" carry the exact same coordinates — or does it
vary row to row for the same city? This has nothing to do with
`distance`; it's purely about whether one column (coordinates) is a
reliable lookup of another (city name).

In [19]:
profiling.check_coordinate_stability(train_test, "pickup", "pickup_lat", "pickup_lon")

INFO src.profiling: Checking coordinate stability for pickup
INFO src.profiling: Coordinate stability (pickup): {'skipped': False, 'n_cities_checked': 64, 'max_lat_std': 0.0, 'max_lon_std': 0.0, 'is_fully_stable': True}


{'skipped': False,
 'n_cities_checked': 64,
 'max_lat_std': 0.0,
 'max_lon_std': 0.0,
 'is_fully_stable': True}

In [20]:
profiling.check_coordinate_stability(train_test, "delivery", "delivery_lat", "delivery_lon")

INFO src.profiling: Checking coordinate stability for delivery
INFO src.profiling: Coordinate stability (delivery): {'skipped': False, 'n_cities_checked': 64, 'max_lat_std': 0.0, 'max_lon_std': 0.0, 'is_fully_stable': True}


{'skipped': False,
 'n_cities_checked': 64,
 'max_lat_std': 0.0,
 'max_lon_std': 0.0,
 'is_fully_stable': True}

Both pickup and delivery coordinates are fully stable (standard
deviation of 0.0) across all 64 cities that appear more than once —
confirming lat/lon is deterministic geocoding by city name, not noisy
per-row measurement. This is a clean result with no follow-up needed.

## Date coverage: train_test vs. validation

What time period does each file actually cover, and how do the two
ranges relate to each other?

In [21]:
tt_coverage = profiling.check_date_coverage(train_test, config.DATE_COL, "train_test")
val_coverage = profiling.check_date_coverage(validation, config.DATE_COL, "validation")

INFO src.profiling: Checking date coverage for train_test
INFO src.profiling: Date coverage (train_test): {'name': 'train_test', 'n_rows': 48000, 'n_unparseable': 0, 'min_date': Timestamp('2025-01-01 00:00:00'), 'max_date': Timestamp('2025-10-31 00:00:00'), 'n_unique_dates': 304}
INFO src.profiling: Checking date coverage for validation
INFO src.profiling: Date coverage (validation): {'name': 'validation', 'n_rows': 12000, 'n_unparseable': 0, 'min_date': Timestamp('2025-11-01 00:00:00'), 'max_date': Timestamp('2025-12-31 00:00:00'), 'n_unique_dates': 61}


In [22]:
pd.DataFrame([tt_coverage, val_coverage]).set_index("name")

,n_rows,n_unparseable,min_date,max_date,n_unique_dates
name,,,,,
train_test,48000,0,2025-01-01,2025-10-31,304
validation,12000,0,2025-11-01,2025-12-31,61


`train_test` runs 2025-01-01 through 2025-10-31 (304 unique dates, 10
months). `validation` runs 2025-11-01 through 2025-12-31 (61 unique
dates, 2 months). Zero unparseable dates in either file.

The two ranges are contiguous with zero overlap and zero gap —
`train_test` ends the day before `validation` begins. This is a
structural fact about the data, not a pattern that required looking
for a relationship: **any split of `train_test` into our own
train/tune/test slices must be chronological, not a random shuffle**,
because the actual task `validation` poses is forecasting a period
entirely after everything we're allowed to train on. A random
K-fold shuffle would let a model see rows from both before and after
any held-out point, which isn't the situation we're being evaluated
in. This conclusion follows directly from the date ranges themselves;
it doesn't require characterizing *how* the data drifts over time
(trend, seasonality, magnitude) — that's what `02_eda.ipynb` looks at,
and it will inform the specific train/tune/test cutoffs, not whether
the split should be chronological in the first place.

## Summary

Structural facts established in this notebook, and what each one
means for the next steps. Nothing below involves the target's
distribution or any correlation with it — those are EDA questions.

**Shape and structure**
- `train_test`: 48,000 rows × 14 columns (includes `posted_rate`).
  `validation`: 12,000 rows × 13 columns (no target — Spotter scores
  this after submission). `validation_template`: 12,000 rows × 2
  columns (`load_id`, blank `predicted_rate`) matching `validation`'s
  id set exactly. `december`: 31 rows × 7 columns, no `load_id` (row
  identity is `date` instead), missing both coordinate pairs,
  `market_index`, and `quote_signal` entirely — not just empty, absent
  as columns. → Any model used to fill `december`'s predictions must
  be trainable on a feature set that excludes those columns.
- All columns are `float64` or `object` (string); `date` is currently
  a string, not yet parsed. Target is continuous → regression problem.

**Completeness**
- Two columns carry missing values in both `train_test` and
  `validation`: `market_index` and `weight`, both under 1% in
  `train_test`, both meaningfully higher in `validation`. `december`
  has no missing values outside the `predicted_rate` column we're
  filling in. → Because every row of `validation` needs a prediction
  regardless of missingness, and consistency between how `train_test`
  and `validation` are treated matters, imputation (not row-dropping)
  is the direction for cleaning — the specific strategy is a cleaning
  decision.

**Uniqueness**
- `load_id` is 100% unique in both `train_test` and `validation`, and
  neither file has any exact duplicate rows. No de-duplication
  required.

**Domain validity**
- Every domain check passes cleanly except one: `weight <= 0` in 292
  `train_test` rows (0.61%) and 145 `validation` rows (1.21%). These
  are largely a separate set of rows from the missing-weight rows
  (only one row has both issues). → What these negative values
  actually represent needs a real look (magnitude comparison against
  valid weights) before deciding how to treat them — that's the first
  thing `02_cleaning.ipynb` does.

**Cardinality**
- 64 pickup cities, 64 delivery cities, only 3 equipment types. → The
  city cardinality is a real cost to weigh in feature engineering
  (one-hot would add 120+ columns against ~48k rows); equipment's low
  cardinality makes one-hot trivial there.

**Scale**
- Numeric columns span very different ranges purely by unit
  (`distance` in the thousands, `market_index`/`quote_signal` under
  4) → any scale-sensitive model needs scaling in the pipeline; which
  scaler and whether a transform is also needed depends on shape,
  which is EDA's job.

**Distance / coordinate consistency**
- `distance` and coordinate-implied great-circle distance correlate
  at 0.9995 overall — typical ratio ~1.15–1.20, consistent with normal
  road-routing inflation over straight-line distance. 22 rows across
  exactly two lanes (Austin↔Lubbock, New Orleans↔Shreveport, both
  directions) all share the identical `distance = 70.0` despite being
  genuinely different real-world separations — reads as a fallback
  value used upstream for these two lanes specifically, not
  measurement noise. → A deliberate, lane-specific decision for
  cleaning, not a blanket rule.
- Coordinates are perfectly stable per city (0.0 standard deviation
  across all 64 repeated cities in both `pickup` and `delivery`) —
  deterministic geocoding, no follow-up needed.

**Date coverage**
- `train_test` (2025-01-01 to 2025-10-31) and `validation`
  (2025-11-01 to 2025-12-31) are contiguous with zero overlap and zero
  gap. → Our own train/tune/test split of `train_test` must be
  chronological, not a random shuffle — this follows directly from the
  date ranges, not from a discovered pattern. Exact cutoffs are an EDA
  / split-design decision, not fixed here.

**Carried into `02_cleaning.ipynb`, in order:**
1. Decide the weight sign-flip question with an actual visual
   comparison (`src/viz.py`), not asserted from profiling alone.
2. Decide imputation strategy for `weight` and `market_index`
   (train-fit only, applied consistently to `validation`).
3. Decide how to treat the two `distance = 70.0` fallback-value lanes.
4. Parse `date` to a real datetime dtype (deterministic, safe
   pre-split).
5. Save cleaned files to `data/processed`.